This notebook prepares the student performance dataset for machine learning by making feature-selection, preprocessing, and feature-engineering decisions based on the findings from the data understanding and EDA stage.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **1. Load the dataset and libraries**

In [2]:
import pandas as pd

In [3]:
df=pd.read_csv("/content/drive/MyDrive/Ml_Assignment/Data/student-mat.csv",sep=";");

In [4]:
df.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


In [5]:
df.shape

(395, 33)

# **2. Features and Target**

### **2.1 Remove the 38 G3 = 0 records**

In [6]:
df = df[df["G3"] != 0].copy()
print("Dataset shape:", df.shape)
print("G3 = 0:", (df["G3"] == 0).sum())

Dataset shape: (357, 33)
G3 = 0: 0


### **2.2 separate target and features**

In [7]:
target = "G3"

X = df.drop(columns=["G3", "G1", "G2"])
y = df["G3"]

print("Features:", X.shape)
print("Target:", y.shape)

Features: (357, 30)
Target: (357,)


### **2.3 Train/Test Split**

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [9]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (285, 30)
X_test: (72, 30)
y_train: (285,)
y_test: (72,)


# **3. Feature Groups and Preprocessing Decisions**

Group the 30 input features by their meaning, rather than relying only on their stored data types. For example, `studytime` contains ordered category codes, while `age` is a numerical measurement.

| Group | Planned treatment | Reason |
|---|---|---|
| Binary categorical | Encode each pair as 0/1 using documented mappings | Each feature has two categories; the mapping does not imply a ranking |
| Nominal categorical | One-hot encode, with handling for unseen categories | Jobs, school-choice reasons, and guardians have no natural order |
| Ordinal | Retain the existing ordered codes initially | The codes preserve category order, but equal numerical differences do not necessarily represent equal real-world differences |
| Numerical | Review transformations and scale where appropriate for the model | Age, recorded failures, and absences represent numerical quantities |

### **Feature Groups**

| Type | Count | Features |
|------|-------|----------|
| **Binary** | 13 | `school`, `sex`, `address`, `famsize`, `Pstatus`, `schoolsup`, `famsup`, `paid`, `activities`, `nursery`, `higher`, `internet`, `romantic` |
| **Nominal** | 4 | `Mjob`, `Fjob`, `reason`, `guardian` |
| **Ordinal** | 10 | `Medu`, `Fedu`, `traveltime`, `studytime`, `famrel`, `freetime`, `goout`, `Dalc`, `Walc`, `health` |
| **Numerical** | 3 | `age`, `failures`, `absences` |


In [10]:
binary_features = [
    "school", "sex", "address", "famsize", "Pstatus",
    "schoolsup", "famsup", "paid", "activities", "nursery",
    "higher", "internet", "romantic"
]

nominal_features = ["Mjob", "Fjob", "reason", "guardian"]

ordinal_features = [
    "Medu", "Fedu", "traveltime", "studytime", "famrel",
    "freetime", "goout", "Dalc", "Walc", "health"
]

numerical_features = ["age", "failures", "absences"]

### **3.1 Binary Feature Encoding**

Binary categorical features are converted into 0/1 values using explicit


In [11]:
binary_mappings = {
    "school": {"GP": 0, "MS": 1},
    "sex": {"F": 0, "M": 1},
    "address": {"R": 0, "U": 1},
    "famsize": {"LE3": 0, "GT3": 1},
    "Pstatus": {"A": 0, "T": 1},
}

yes_no_features = [
    "schoolsup", "famsup", "paid", "activities",
    "nursery", "higher", "internet", "romantic"
]

for column in yes_no_features:
    binary_mappings[column] = {"no": 0, "yes": 1}

X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()

for column, mapping in binary_mappings.items():
    X_train_encoded[column] = X_train[column].map(mapping)
    X_test_encoded[column] = X_test[column].map(mapping)

X_train_encoded[list(binary_mappings)].head()

,school,sex,address,famsize,Pstatus,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic
308,0,1,0,1,1,0,0,0,1,1,1,0,1
368,1,0,1,1,1,0,1,1,0,1,1,1,1
315,0,0,0,1,1,0,0,0,0,1,1,1,1
75,0,1,1,1,1,0,1,1,1,1,1,1,0
280,0,1,1,0,0,0,0,1,1,1,1,1,1


### **3.2 Nominal Feature Encoding**

Mjob, Fjob, reason, and guardian have unordered categories.
One-hot encoding represents them using binary indicator columns.

In [12]:
from sklearn.preprocessing import OneHotEncoder

nominal_features = ["Mjob", "Fjob", "reason", "guardian"]

nominal_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

train_nominal = nominal_encoder.fit_transform(
    X_train_encoded[nominal_features]
)

test_nominal = nominal_encoder.transform(
    X_test_encoded[nominal_features]
)

In [13]:
encoded_names = nominal_encoder.get_feature_names_out(
    nominal_features
)

train_nominal_df = pd.DataFrame(
    train_nominal,
    columns=encoded_names,
    index=X_train_encoded.index
)

test_nominal_df = pd.DataFrame(
    test_nominal,
    columns=encoded_names,
    index=X_test_encoded.index
)

X_train_encoded = pd.concat(
    [
        X_train_encoded.drop(columns=nominal_features),
        train_nominal_df
    ],
    axis=1
)

X_test_encoded = pd.concat(
    [
        X_test_encoded.drop(columns=nominal_features),
        test_nominal_df
    ],
    axis=1
)

In [14]:
ordinal_allowed_values = {
    "Medu": range(0, 5),
    "Fedu": range(0, 5),
    "traveltime": range(1, 5),
    "studytime": range(1, 5),
    "famrel": range(1, 6),
    "freetime": range(1, 6),
    "goout": range(1, 6),
    "Dalc": range(1, 6),
    "Walc": range(1, 6),
    "health": range(1, 6),
}

for column, allowed_values in ordinal_allowed_values.items():
    assert X_train_encoded[column].isin(allowed_values).all(), column
    assert X_test_encoded[column].isin(allowed_values).all(), column

print("All ordinal values are within their expected categories.")

All ordinal values are within their expected categories.


### **3.4 Absence Transformation**

In [15]:
X_train["absences"].describe()

,absences
count,285.000000
mean,6.284211
std,8.536578
min,0.000000
25%,2.000000
50%,4.000000
75%,8.000000
max,75.000000


Training-set absences have a mean of 6.28, median of 4, and maximum
of 75, suggesting a right-skewed distribution.

High-absence observations are retained because there is no evidence
that they are recording errors. A log1p transformation is created
as a candidate representation to compress large values while
preserving zero absences and the ordering of observations.

Its predictive usefulness will be compared with the original
representation through training cross-validation.

In [16]:
import numpy as np


X_train_log = X_train_encoded.copy()
X_test_log = X_test_encoded.copy()

X_train_log["absences_log"] = np.log1p(X_train_log["absences"])
X_test_log["absences_log"] = np.log1p(X_test_log["absences"])

X_train_log = X_train_log.drop(columns="absences")
X_test_log = X_test_log.drop(columns="absences")

In [17]:
pd.DataFrame({
    "Original absences": X_train_encoded["absences"],
    "Log-transformed absences": X_train_log["absences_log"]
}).describe()

,Original absences,Log-transformed absences
count,285.000000,285.000000
mean,6.284211,1.491478
std,8.536578,1.021282
min,0.000000,0.000000
25%,2.000000,1.098612
50%,4.000000,1.609438
75%,8.000000,2.197225
max,75.000000,4.330733


### **3.5 Feature Scaling**

Numerical and ordinal features are standardized for models that
benefit from comparable feature scales, such as Ridge Regression.

Binary and one-hot encoded features remain unchanged in this
initial preprocessing design.

In [18]:
from sklearn.preprocessing import StandardScaler

scale_features = ordinal_features + [
    "age", "failures", "absences_log"
]

scaler = StandardScaler()

X_train_scaled = X_train_log.copy()
X_test_scaled = X_test_log.copy()

X_train_scaled[scale_features] = scaler.fit_transform(
    X_train_log[scale_features]
)

X_test_scaled[scale_features] = scaler.transform(
    X_test_log[scale_features]
)

In [19]:
pd.DataFrame({
    "Mean after scaling": X_train_scaled[scale_features].mean(),
    "Std after scaling": X_train_scaled[scale_features].std(ddof=0)
}).round(3)

,Mean after scaling,Std after scaling
Medu,-0.0,1.0
Fedu,0.0,1.0
traveltime,-0.0,1.0
studytime,-0.0,1.0
famrel,0.0,1.0
freetime,0.0,1.0
goout,0.0,1.0
Dalc,-0.0,1.0
Walc,-0.0,1.0
health,-0.0,1.0


In [20]:
assert X_train_scaled.index.equals(y_train.index)
assert X_test_scaled.index.equals(y_test.index)

assert X_train_scaled.columns.equals(X_test_scaled.columns)

assert X_train_scaled.isna().sum().sum() == 0
assert X_test_scaled.isna().sum().sum() == 0

assert np.isfinite(X_train_scaled.to_numpy()).all()
assert np.isfinite(X_test_scaled.to_numpy()).all()

assert not {"G1", "G2", "G3"}.intersection(X_train_scaled.columns)

print("Preprocessing checks passed.")
print("Training shape:", X_train_scaled.shape)
print("Test shape:", X_test_scaled.shape)

Preprocessing checks passed.
Training shape: (285, 43)
Test shape: (72, 43)


#**preprocessing decisions and their reasons**

| Item | Decision | Reason / limitation |
|---|---|---|
| Dataset scope | Use Mathematics data | Keeps prediction focused on one subject. |
| Zero final grades | Exclude 38 records with `G3 = 0` | Focuses the task on positive recorded final grades. The model therefore does not cover all student outcomes; zero grades are not confirmed missing values or withdrawals. |
| Grade features | Exclude `G1` and `G2` | Supports prediction without prior-period grades. Availability of the other features at prediction time still requires review. |
| Missing values | No imputation currently needed | Data-quality checks found no missing values. |
| Duplicate records | No removal needed | Data-quality checks found no exact duplicate rows. |
| Binary features | Apply explicit 0/1 mappings | Represents two-category features consistently. |
| Nominal features | One-hot encode using training categories | Avoids imposing an artificial order; unseen categories are handled with `handle_unknown="ignore"`. |
| Ordinal features | Retain existing ordered codes initially | Preserves order. Linear models assume equal effects per code step, which may not reflect the underlying categories. |
| Extreme absences | Retain observations | Large values alone do not establish recording errors. |
| Absence transformation | Compare original absences with `log1p(absences)` | The log representation compresses the upper tail. Improvement remains unverified until training cross-validation. |
| Scaling | Standardize numerical and ordinal inputs for Ridge; leave binary indicators unchanged initially | Makes scales comparable for regularization. Tree models will use unscaled inputs. |
| Rare categories | Retain initially | Small category counts alone do not justify merging meaningful categories. |
| Validation | Fit learned preprocessing within each cross-validation training fold | Prevents validation-fold information from influencing encoding or scaling. |

# **5. Reusable Preprocessing Pipeline**

Preprocessing is combined into a reusable pipeline operating on the
original input features.

During model comparison, this preprocessing pipeline will be joined
to each model so learned transformations are fitted separately
within each cross-validation training fold.

In [21]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def transform_absences(X, use_log=False):
    X = X.copy()

    if use_log:
        if (X["absences"] < 0).any():
            raise ValueError("Absences cannot be negative.")

        X["absences_log"] = np.log1p(X["absences"])
        X = X.drop(columns="absences")

    return X


def encode_binary(X):
    X = X.copy()

    for column in X.columns:
        mapping = binary_mappings[column]

        if not X[column].isin(mapping.keys()).all():
            raise ValueError(
                f"Missing or unexpected category in {column}"
            )

        X[column] = X[column].map(mapping)

    return X

In [24]:
def make_preprocessor(use_log_absences=False, scale_numeric=False):
    absence_feature = (
        "absences_log" if use_log_absences else "absences"
    )

    numeric_and_ordinal = ordinal_features + [
        "age", "failures", absence_feature
    ]

    numeric_transformer = (
        StandardScaler() if scale_numeric else "passthrough"
    )

    column_processing = ColumnTransformer(
        transformers=[
            (
                "binary",
                FunctionTransformer(
                    encode_binary,
                    validate=False,
                    feature_names_out="one-to-one"
                ),
                binary_features
            ),
            (
                "nominal",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                nominal_features
            ),
            (
                "numeric_ordinal",
                numeric_transformer,
                numeric_and_ordinal
            )
        ],
        remainder="drop"
    )

    return Pipeline([
        (
            "absence_transform",
            FunctionTransformer(
                transform_absences,
                kw_args={"use_log": use_log_absences},
                validate=False
            )
        ),
        ("column_processing", column_processing)
    ])

In [25]:
preprocessor_demo = make_preprocessor(
    use_log_absences=True,
    scale_numeric=True
)

X_train_demo = preprocessor_demo.fit_transform(X_train)
X_test_demo = preprocessor_demo.transform(X_test)

assert X_train_demo.shape[0] == len(y_train)
assert X_test_demo.shape[0] == len(y_test)
assert X_train_demo.shape[1] == X_test_demo.shape[1]

assert np.isfinite(X_train_demo).all()
assert np.isfinite(X_test_demo).all()

print("Training shape:", X_train_demo.shape)
print("Test shape:", X_test_demo.shape)
print("Pipeline checks passed.")

Training shape: (285, 43)
Test shape: (72, 43)
Pipeline checks passed.
